In [1]:
import os
import json
import numpy as np
import pandas as pd
from scipy import stats

# ==========================================
# 1. 路径与配置 (Parler SUM)
# ==========================================
parent_dataset = "parler_data"
dataset_name = "dataset_test"   # 或 dataset_three
base_dir = f"/home/wangshuo/resource/datasets/{parent_dataset}/{dataset_name}/results"

# 包含 8_POSSA 及其他对比方法的 CSV 文件路径
alloc_csv_path = os.path.join(base_dir, "efficiency", "allocation_strategy_comparison_sum.csv")
if not os.path.exists(alloc_csv_path):
    alloc_csv_path = os.path.join(base_dir, "efficiency", "allocation_strategy_comparison_ablation_sum.csv")

# Ground Truth JSON 路径
gt_json_path = os.path.join(base_dir, "T_true_ML1_oracle2_probability_ML2_oracle2_probability_sum.json")

TARGET_FRAC = 0.1       # 目标采样率 10%
OUR_METHOD = "8_POSSA"  # 你的核心方法名

# ==========================================
# 2. 读取真值与数据加载
# ==========================================
if not os.path.exists(alloc_csv_path):
    raise FileNotFoundError(f"❌ 找不到结果文件: {alloc_csv_path}")
if not os.path.exists(gt_json_path):
    raise FileNotFoundError(f"❌ 找不到 Ground Truth 文件: {gt_json_path}")

with open(gt_json_path, 'r', encoding='utf-8') as f:
    gt_dict = json.load(f)
gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None and v > 0}

df = pd.read_csv(alloc_csv_path)

# 过滤指定采样率
if "budget_frac" in df.columns:
    df = df[np.isclose(df["budget_frac"], TARGET_FRAC, atol=1e-4)].copy()

df["query_clean"] = df["query_basename"].astype(str).str.replace(r"\.graph$", "", regex=True)

# 对齐 Ground Truth 并计算每一轮单次的 ARE
epsilon = 1e-9
df["T_true_matched"] = df["query_clean"].map(gt_map)
df = df.dropna(subset=["T_true_matched"]).copy()
df["ARE"] = (df["T_hat"] - df["T_true_matched"]).abs() / (df["T_true_matched"] + epsilon)

print(f"[*] 成功加载数据: 包含 {df['query_clean'].nunique()} 个查询，涵盖方法: {df['method'].unique().tolist()}\n")

# ==========================================
# 3. 检验指标一：Per-Query 稳定性与标准差 σ 检验
# ==========================================
df_our = df[df["method"].isin([OUR_METHOD, "POSS"])].copy()

# 按 query 计算多轮估计值的统计量
query_stats = df_our.groupby("query_clean").agg(
    run_count=("T_hat", "count"),
    mean_that=("T_hat", "mean"),
    std_that=("T_hat", "std"),
    t_true=("T_true_matched", "first"),
    mean_are=("ARE", "mean"),
    std_are=("ARE", "std")
).reset_index()

# 计算相对标准差 (RSD / CV = std(T_hat) / T_true)
query_stats["relative_sigma"] = query_stats["std_that"] / (query_stats["t_true"] + epsilon)

avg_sigma = query_stats["relative_sigma"].mean()
median_sigma = query_stats["relative_sigma"].median()
max_sigma = query_stats["relative_sigma"].max()

# 统计满足 sigma < 2% (0.02) 的查询比例
sigma_threshold = 0.02
satisfied_count = (query_stats["relative_sigma"] < sigma_threshold).sum()
satisfied_ratio = satisfied_count / len(query_stats) * 100

print("=" * 70)
print(f"📊 [指标 1] 单查询稳定性检验: {OUR_METHOD} (多轮运行 σ 分析)")
print("=" * 70)
print(f"1. 平均每个查询运行轮数 (Runs)      : {query_stats['run_count'].mean():.1f} 轮")
print(f"2. 单查询相对标准差均值 (Mean σ)     : {avg_sigma:.4f} ({avg_sigma * 100:.2f}%)")
print(f"3. 单查询相对标准差中位数 (Med σ)    : {median_sigma:.4f} ({median_sigma * 100:.2f}%)")
print(f"4. 单查询相对标准差最大值 (Max σ)    : {max_sigma:.4f} ({max_sigma * 100:.2f}%)")
print(f"5. 满足 σ < {sigma_threshold*100:.0f}% 要求的查询比例 : {satisfied_count}/{len(query_stats)} ({satisfied_ratio:.2f}%)")
print("=" * 70 + "\n")

# ==========================================
# 4. 检验指标二：配对显著性检验 (Paired t-test & Wilcoxon)
# ==========================================
# 计算每个方法在每个 query 上的平均 ARE
query_method_are = df.groupby(["query_clean", "method"])["ARE"].mean().unstack()

print("=" * 80)
print(f"📊 [指标 2] 统计显著性检验: {OUR_METHOD} vs 其他基线 (Paired Tests)")
print("=" * 80)
print(f"{'对比基线 (Baseline)':<22} | {'POSS Mean ARE':<14} | {'Base Mean ARE':<14} | {'t-stat':<10} | {'p-value (t-test)':<18} | {'显著性 (p<0.05)?'}")
print("-" * 80)

our_col = OUR_METHOD if OUR_METHOD in query_method_are.columns else "POSS"
our_ares = query_method_are[our_col]

for method in query_method_are.columns:
    if method == our_col:
        continue
    
    # 提取共有 query 的成对样本
    paired_df = query_method_are[[our_col, method]].dropna()
    if len(paired_df) < 5:
        continue
    
    x = paired_df[our_col].values
    y = paired_df[method].values
    
    # 1. 配对样本 t 检验 (Paired t-test)
    t_stat, p_val_t = stats.ttest_rel(x, y, alternative='less') # 'less' 代表检验 POSS 是否显著小于基线
    
    # 2. 配对非参数检验 (Wilcoxon Signed-Rank Test，对非正态数据更稳健)
    try:
        w_stat, p_val_w = stats.wilcoxon(x, y, alternative='less')
    except Exception:
        p_val_w = np.nan

    is_sig = "✅ 显著 (p < 0.05)" if p_val_t < 0.05 else "❌ 不显著"
    
    print(f"{method:<22} | {x.mean()*100:>11.2f}% | {y.mean()*100:>11.2f}% | {t_stat:>9.2f} | {p_val_t:>16.4e} | {is_sig}")

print("=" * 80)

[*] 成功加载数据: 包含 110 个查询，涵盖方法: ['8_POSSA']

📊 [指标 1] 单查询稳定性检验: 8_POSSA (多轮运行 σ 分析)
1. 平均每个查询运行轮数 (Runs)      : 5.0 轮
2. 单查询相对标准差均值 (Mean σ)     : 0.0106 (1.06%)
3. 单查询相对标准差中位数 (Med σ)    : 0.0062 (0.62%)
4. 单查询相对标准差最大值 (Max σ)    : 0.0788 (7.88%)
5. 满足 σ < 2% 要求的查询比例 : 96/110 (87.27%)

📊 [指标 2] 统计显著性检验: 8_POSSA vs 其他基线 (Paired Tests)
对比基线 (Baseline)        | POSS Mean ARE  | Base Mean ARE  | t-stat     | p-value (t-test)   | 显著性 (p<0.05)?
--------------------------------------------------------------------------------


In [2]:
import os
import json
import numpy as np
import pandas as pd
from scipy import stats

# ==========================================
# 1. 路径与配置 (支持随时切换 Parler 或 Amazon)
# ==========================================
# --- Parler-E 配置 ---
parent_dataset = "parler_data"
dataset_name = "dataset_test"
gt_filename = "T_true_ML1_oracle2_probability_ML2_oracle2_probability_sum.json"

# --- Amazon 配置 (如需测试 Amazon，请取消下面3行注释) ---
# parent_dataset = "amazon_data"
# dataset_name = "amazon_extend"
# gt_filename = "T_true_ML3_oracle2_probability_ML2_oracle1_probability_sum.json"

base_dir = f"/home/wangshuo/resource/datasets/{parent_dataset}/{dataset_name}/results"
eff_dir = os.path.join(base_dir, "efficiency")

# 两个对比 CSV 路径
path_poss = os.path.join(eff_dir, "allocation_strategy_comparison_sum.csv")
if not os.path.exists(path_poss):
    path_poss = os.path.join(eff_dir, "allocation_strategy_comparison_ablation_sum.csv")

path_fastesto = os.path.join(eff_dir, "FastestO_budget_curve_sum.csv")

# Ground Truth JSON 路径
path_gt = os.path.join(base_dir, gt_filename)

TARGET_FRAC = 0.1  # 目标采样率 10%

# ==========================================
# 2. 数据加载与对齐函数
# ==========================================
def load_and_preprocess():
    print(f"[*] 正在加载数据集: {dataset_name} (采样率 Budget = {int(TARGET_FRAC*100)}%)")
    
    if not os.path.exists(path_poss):
        raise FileNotFoundError(f"❌ 找不到 POSS 结果文件: {path_poss}")
    if not os.path.exists(path_fastesto):
        raise FileNotFoundError(f"❌ 找不到 FaSTestO 结果文件: {path_fastesto}")
    if not os.path.exists(path_gt):
        raise FileNotFoundError(f"❌ 找不到 Ground Truth 文件: {path_gt}")

    # 读取 Ground Truth
    with open(path_gt, 'r', encoding='utf-8') as f:
        gt_dict = json.load(f)
    gt_map = {str(k).replace(".graph", ""): float(v) for k, v in gt_dict.items() if v is not None and v > 0}
    print(f"[+] 成功加载 Ground Truth，包含 {len(gt_map)} 个有效查询。")

    def process_df(csv_path, target_methods, label_name):
        df = pd.read_csv(csv_path)
        # 过滤指定采样率
        if "budget_frac" in df.columns:
            df = df[np.isclose(df["budget_frac"], TARGET_FRAC, atol=1e-4)].copy()
        
        # 过滤方法名
        df = df[df["method"].isin(target_methods)].copy()
        df["query_clean"] = df["query_basename"].astype(str).str.replace(r"\.graph$", "", regex=True)
        
        # 匹配 GT
        df["T_true_matched"] = df["query_clean"].map(gt_map)
        df = df.dropna(subset=["T_true_matched"]).copy()
        
        # 计算单次运行的 ARE 与 Signed RE
        eps = 1e-9
        df["ARE"] = (df["T_hat"] - df["T_true_matched"]).abs() / (df["T_true_matched"] + eps)
        df["Signed_RE"] = (df["T_hat"] - df["T_true_matched"]) / (df["T_true_matched"] + eps)
        df["method_standard"] = label_name
        return df

    df_poss = process_df(path_poss, ["8_POSSA", "POSS"], "POSS (8_POSSA)")
    df_fast = process_df(path_fastesto, ["FastestO", "FaSTestO"], "FaSTestO")

    return df_poss, df_fast, gt_map

# ==========================================
# 3. 执行对比分析与输出
# ==========================================
def main():
    df_poss, df_fast, gt_map = load_and_preprocess()

    # 提取公共查询
    common_queries = sorted(list(set(df_poss["query_clean"].unique()) & set(df_fast["query_clean"].unique())))
    print(f"[+] 双方共同包含的查询总数 (交集): {len(common_queries)} 个\n")

    df_poss_comm = df_poss[df_poss["query_clean"].isin(common_queries)].copy()
    df_fast_comm = df_fast[df_fast["query_clean"].isin(common_queries)].copy()

    # ---------------------------------------------------------
    # 报表 1：单次运行视角综合统计表 (VLDB / ICDE 期刊标准格式)
    # ---------------------------------------------------------
    def get_metrics_row(df_sub, name):
        return {
            "Method": name,
            "Runs": len(df_sub),
            "Mean ARE": f"{df_sub['ARE'].mean() * 100:.2f}%",
            "Median ARE": f"{df_sub['ARE'].median() * 100:.2f}%",
            "P70 ARE": f"{df_sub['ARE'].quantile(0.70) * 100:.2f}%",
            "P90 ARE": f"{df_sub['ARE'].quantile(0.90) * 100:.2f}%",
            "P95 ARE": f"{df_sub['ARE'].quantile(0.95) * 100:.2f}%",
            "Max ARE": f"{df_sub['ARE'].max() * 100:.2f}%",
            "Signed RE": f"{df_sub['Signed_RE'].mean() * 100:.2f}%",
            "Avg Oracle Cost": f"{df_sub['oracle_cost'].mean():.1f}" if "oracle_cost" in df_sub.columns else "N/A"
        }

    summary_rows = [
        get_metrics_row(df_poss_comm, "POSS (Our Method)"),
        get_metrics_row(df_fast_comm, "FaSTestO (Baseline)")
    ]
    summary_df = pd.DataFrame(summary_rows)

    print("=" * 110)
    print(f"📊 【表 1】端到端精度与开销对比表 (SUM Aggregation, Budget = {int(TARGET_FRAC*100)}%)")
    print("=" * 110)
    print(summary_df.to_string(index=False))
    print("=" * 110 + "\n")

    # ---------------------------------------------------------
    # 报表 2：逐查询配对显著性检验 (Paired Significance Tests)
    # ---------------------------------------------------------
    # 按 query 求平均 ARE，消除单轮随机波动对检验的影响
    poss_query_are = df_poss_comm.groupby("query_clean")["ARE"].mean().loc[common_queries].values
    fast_query_are = df_fast_comm.groupby("query_clean")["ARE"].mean().loc[common_queries].values

    # 1. 配对样本 t 检验 (Paired t-test, 单尾检验: POSS < FaSTestO)
    t_stat, p_val_t = stats.ttest_rel(poss_query_are, fast_query_are, alternative='less')

    # 2. 配对 Wilcoxon 符号秩检验 (非参数检验，对长尾极值更稳健)
    w_stat, p_val_w = stats.wilcoxon(poss_query_are, fast_query_are, alternative='less')

    # 3. 逐查询胜率统计
    wins = np.sum(poss_query_are < fast_query_are)
    ties = np.sum(np.isclose(poss_query_are, fast_query_are, atol=1e-5))
    losses = len(common_queries) - wins - ties
    win_rate = (wins / len(common_queries)) * 100

    # 4. 平均相对误差降低幅度 (Relative Error Reduction)
    reduction = ((fast_query_are.mean() - poss_query_are.mean()) / fast_query_are.mean()) * 100

    print("=" * 80)
    print("📈 【表 2】配对统计显著性检验与胜率分析 (POSS vs FaSTestO)")
    print("=" * 80)
    print(f"1. 共同对齐查询总数 (Sample Size N)  : {len(common_queries)}")
    print(f"2. POSS 胜出查询数 (Wins / Total)     : {wins} / {len(common_queries)} (胜率: {win_rate:.2f}%)")
    print(f"3. 平局 / 落后查询数 (Ties / Losses) : {ties} / {losses}")
    print(f"4. 均值相对误差降幅 (Error Reduction): {reduction:.2f}% (误差相对缩小了 {reduction:.1f}%)")
    print("-" * 80)
    print(f"5. 配对 t 检验统计量 (Paired t-stat)  : {t_stat:.4f}")
    print(f"   -> t-test p-value                 : {p_val_t:.4e} {'(*** p < 0.001 极显著)' if p_val_t < 0.001 else ''}")
    print(f"6. Wilcoxon 符号秩检验统计量 (W-stat): {w_stat:.1f}")
    print(f"   -> Wilcoxon p-value               : {p_val_w:.4e} {'(*** p < 0.001 极显著)' if p_val_w < 0.001 else ''}")
    print("=" * 80 + "\n")

    # ---------------------------------------------------------
    # 论文撰写英文总结 (可直接复制进 LaTeX)
    # ---------------------------------------------------------
    print("📝 【论文写作建议（直接复制到 Paper 中）】:")
    print("-" * 80)
    sig_text = "p < 10^{-5}" if p_val_t < 1e-5 else f"p = {p_val_t:.2e}"
    print(f"> \"Under the same Oracle budget (10%), POSS significantly outperforms FaSTestO, "
          f"reducing the Mean ARE from {df_fast_comm['ARE'].mean()*100:.2f}% to {df_poss_comm['ARE'].mean()*100:.2f}% "
          f"(a {reduction:.1f}% relative error reduction). "
          f"A paired t-test across all {len(common_queries)} queries confirms that the improvement "
          f"is statistically significant ({sig_text}, t = {t_stat:.2f}), "
          f"with POSS achieving lower error on {win_rate:.1f}% of the evaluation queries.\"\n")

if __name__ == "__main__":
    main()

[*] 正在加载数据集: dataset_test (采样率 Budget = 10%)
[+] 成功加载 Ground Truth，包含 110 个有效查询。
[+] 双方共同包含的查询总数 (交集): 110 个

📊 【表 1】端到端精度与开销对比表 (SUM Aggregation, Budget = 10%)
             Method  Runs Mean ARE Median ARE P70 ARE P90 ARE P95 ARE Max ARE Signed RE Avg Oracle Cost
  POSS (Our Method)   550    7.73%      4.61%   7.83%  20.95%  27.07%  45.46%    -1.43%           731.2
FaSTestO (Baseline)   220   40.51%     28.67%  48.22%  88.02% 100.00% 400.48%    -9.89%           877.8

📈 【表 2】配对统计显著性检验与胜率分析 (POSS vs FaSTestO)
1. 共同对齐查询总数 (Sample Size N)  : 110
2. POSS 胜出查询数 (Wins / Total)     : 106 / 110 (胜率: 96.36%)
3. 平局 / 落后查询数 (Ties / Losses) : 0 / 4
4. 均值相对误差降幅 (Error Reduction): 80.92% (误差相对缩小了 80.9%)
--------------------------------------------------------------------------------
5. 配对 t 检验统计量 (Paired t-stat)  : -9.7223
   -> t-test p-value                 : 9.1953e-17 (*** p < 0.001 极显著)
6. Wilcoxon 符号秩检验统计量 (W-stat): 32.0
   -> Wilcoxon p-value               : 1.0488e-19 (*** p < 0.001 极显著)

📝